In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tqdm.auto import tqdm

In [9]:
# ==========================
# Configuration
# ==========================

DATASET_PATH = "/content/drive/MyDrive/data/logmel_dataset"

METADATA_PATH = os.path.join(DATASET_PATH, "metadata.csv")

BATCH_SIZE = 64

LEARNING_RATE = 1e-3

EPOCHS = 20

IMAGE_SIZE = 128

SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

Device : cpu


In [10]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [11]:
DATASET_ROOT = "/content/drive/MyDrive/data/logmel_dataset"

METADATA_PATH = os.path.join(
    DATASET_ROOT,
    "metadata.csv"
)

LABEL_MAP_PATH = os.path.join(
    DATASET_ROOT,
    "label_map.json"
)

print(METADATA_PATH)
print(LABEL_MAP_PATH)

/content/drive/MyDrive/data/logmel_dataset/metadata.csv
/content/drive/MyDrive/data/logmel_dataset/label_map.json


In [12]:
metadata = pd.read_csv(METADATA_PATH)

print("Shape :", metadata.shape)

metadata.head()

Shape : (34861, 2)


,filepath,label
0,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
1,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
2,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
3,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
4,/content/drive/MyDrive/data/logmel_dataset/cv4...,0


In [13]:
print("\nColumns:")
print(metadata.columns)

print("\nMissing Values:")
print(metadata.isnull().sum())

print("\nNumber of Classes:")
print(metadata["label"].nunique())

print("\nSamples per Class:")
print(metadata["label"].value_counts())


Columns:
Index(['filepath', 'label'], dtype='object')

Missing Values:
filepath    0
label       0
dtype: int64

Number of Classes:
25

Samples per Class:
label
8     1406
0     1394
1     1394
3     1394
2     1394
4     1394
5     1394
6     1394
7     1394
9     1394
10    1394
11    1394
12    1394
13    1394
14    1394
15    1394
17    1394
21    1394
18    1394
19    1394
20    1394
23    1394
22    1394
24    1394
16    1393
Name: count, dtype: int64


In [14]:
label_encoder = LabelEncoder()

metadata["label"] = label_encoder.fit_transform(
    metadata["label"]
)

NUM_CLASSES = len(label_encoder.classes_)

print("Number of Classes :", NUM_CLASSES)

metadata.head()

Number of Classes : 25


,filepath,label
0,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
1,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
2,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
3,/content/drive/MyDrive/data/logmel_dataset/cv4...,0
4,/content/drive/MyDrive/data/logmel_dataset/cv4...,0


In [15]:
import pickle

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Label encoder saved successfully.")

Label encoder saved successfully.


In [16]:
train_df, val_df = train_test_split(
    metadata,
    test_size=0.2,
    random_state=SEED,
    stratify=metadata["label"]
)

print("Training Samples :", len(train_df))
print("Validation Samples :", len(val_df))

Training Samples : 27888
Validation Samples : 6973


In [8]:
metadata = pd.read_csv("/content/drive/MyDrive/data/logmel_dataset/metadata.csv")

bad_path = "/content/drive/MyDrive/data/logmel_dataset/tf_ljspeech_FASTSPEECH_MB-MELGAN/3a8a5cf359a9746cd6a72bc98df5b953.npy"

metadata = metadata[metadata["filepath"] != bad_path]

metadata.to_csv(
    "/content/drive/MyDrive/data/logmel_dataset/metadata.csv",
    index=False
)

print(len(metadata))

34861


In [17]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_df.head()

,filepath,label
0,/content/drive/MyDrive/data/logmel_dataset/vct...,22
1,/content/drive/MyDrive/data/logmel_dataset/vct...,23
2,/content/drive/MyDrive/data/logmel_dataset/vct...,21
3,/content/drive/MyDrive/data/logmel_dataset/tf_...,19
4,/content/drive/MyDrive/data/logmel_dataset/vct...,21


In [18]:
class SpeechDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        # File path
        path = self.data.iloc[idx]["filepath"]

        # Label
        label = self.data.iloc[idx]["label"]

        # Load spectrogram
        try:
          spec = np.load(path)
        except Exception as e:
          print(f"Error loading file {path}: {e}")
          raise e

        # Add channel dimension
        # (128,128) -> (1,128,128)
        spec = np.expand_dims(spec, axis=0)

        # Convert to tensor
        spec = torch.tensor(spec, dtype=torch.float32)

        label = torch.tensor(label, dtype=torch.long)

        return spec, label

In [20]:
train_dataset = SpeechDataset(train_df)

val_dataset = SpeechDataset(val_df)

print("Training Samples :", len(train_dataset))
print("Validation Samples :", len(val_dataset))

Training Samples : 27888
Validation Samples : 6973


In [21]:
spec, label = train_dataset[0]

print(spec.shape)
print(spec.dtype)
print(label)

torch.Size([1, 128, 128])
torch.float32
tensor(22)


In [22]:

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [23]:
images, labels = next(iter(train_loader))

print("Images Shape :", images.shape)
print("Labels Shape :", labels.shape)

print(images.dtype)
print(labels.dtype)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Images Shape : torch.Size([64, 1, 128, 128])
Labels Shape : torch.Size([64])
torch.float32
torch.int64


In [24]:

class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        # -------- Feature Extractor --------

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # -------- Classifier --------

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(
            64 * 32 * 32,
            256
        )

        self.fc2 = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):

        # Block 1
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        # Block 2
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        # Flatten
        x = self.flatten(x)

        # Fully Connected
        x = self.fc1(x)
        x = self.relu(x)

        # Output logits
        x = self.fc2(x)

        return x

In [25]:

model = CNN(num_classes=NUM_CLASSES).to(DEVICE)

print(model)

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=65536, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=25, bias=True)
)


In [26]:
images, labels = next(iter(train_loader))

images = images.to(DEVICE)

outputs = model(images)

print("Input Shape :", images.shape)
print("Output Shape:", outputs.shape)

Input Shape : torch.Size([64, 1, 128, 128])
Output Shape: torch.Size([64, 25])


In [27]:
criterion = nn.CrossEntropyLoss()

In [28]:

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [29]:

def calculate_accuracy(outputs, labels):

    predictions = torch.argmax(outputs, dim=1)

    correct = (predictions == labels).sum().item()

    accuracy = correct / labels.size(0)

    return accuracy

In [30]:
images, labels = next(iter(train_loader))

images = images.to(DEVICE)
labels = labels.to(DEVICE)

outputs = model(images)

acc = calculate_accuracy(outputs, labels)

print(acc)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


0.0


In [31]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    running_accuracy = 0.0

    progress_bar = tqdm(dataloader)

    for images, labels in progress_bar:


        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()


        running_loss += loss.item()
        running_accuracy += calculate_accuracy(outputs, labels)

        progress_bar.set_description(
            f"Loss: {loss.item():.4f}"
        )

    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = running_accuracy / len(dataloader)

    return epoch_loss, epoch_accuracy

In [32]:
def validate_one_epoch(model, dataloader, criterion, device):

    model.eval()

    running_loss = 0.0
    running_accuracy = 0.0

    with torch.no_grad():

        for images, labels in dataloader:

            # Move data to GPU
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Compute loss
            loss = criterion(outputs, labels)

            # Statistics
            running_loss += loss.item()
            running_accuracy += calculate_accuracy(outputs, labels)

    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = running_accuracy / len(dataloader)

    return epoch_loss, epoch_accuracy

In [33]:
import time

best_val_acc = 0.0

for epoch in range(20):

    start = time.time()

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    val_loss, val_acc = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Best model saved")

  0%|          | 0/436 [00:00<?, ?it/s]

Epoch [1/20] | Train Loss: 1.2033 | Train Acc: 0.6255 | Val Loss: 0.3543 | Val Acc: 0.8787 | Time: 16290.8s
✅ Best model saved


  0%|          | 0/436 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [1]:
import torch

print(torch.cuda.get_device_name(0))

print(torch.cuda.memory_allocated()/1024**2)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
torch.save(model.state_dict(), "cnn_speaker_classifier.pth")

print("Model saved successfully.")